In [ ]:
import geopandas as gpd
from datetime import datetime
from swmm_api import SwmmInput, swmm5_run
from swmm_api.input_file.section_labels import *
from swmm_api.input_file.sections import *

In [ ]:
nodes = gpd.read_file('nodes.geojson')
edges = gpd.read_file('edges.geojson')
subcatchments = gpd.read_file('subcatchments.geojson')

In [ ]:
nodes['SWMM_ID'] = [f"JN_{i}" if t == 'junction' else f"OF_{i}" if t=='outfall' else f"IN_{i}" for i, t in enumerate(nodes['node_type'])]
subcatchments['SWMM_ID'] = [f"SUB_{i}" for i in range(len(subcatchments))]
edges['SWMM_ID'] = [f"CON_{i}" for i in range(len(edges))]

node_to_id = {row.node_id: row.SWMM_ID for _, row in nodes.iterrows()}

In [ ]:
inp = SwmmInput()

inp[OPTIONS] = OptionSection(
    FLOW_UNITS='CMS',
    INFILTRATION='GREEN_AMPT',
    FLOW_ROUTING='DYNWAVE',
    START_DATE='01/01/2025',
    START_TIME='00:00:00',
    END_DATE='01/02/2025',
    END_TIME='00:00:00',
)

In [ ]:
inp[RAINGAGES]['RG1'] = RainGage(
    name='RG1',
    form='INTENSITY',
    interval='0:15:00',
    source='TIMESERIES',
    timeseries='TS1',
    SCF=1.0
)

inp[TIMESERIES]['TS1'] = TimeseriesData(
    name='TS1',
    data=[('00:00', 0.0), ('00:15', 1.0), ('00:30', 0.5), ('00:45', 0.25), ('01:00', 0.0)]
)

for i, row in subcatchments.iterrows():
    area = row.geometry.area
    width = area ** 0.5
    area /= 10_000 # m^2 to hectares
    inp[SUBCATCHMENTS][row.SWMM_ID] = SubCatchment(
        name=row.SWMM_ID,
        rain_gage='RG1', # TODO: split county into rain gages and assign subcatchments accordingly
        outlet=node_to_id[row.node_id],
        area=area,
        width=width,
        slope=row.slope,
        imperviousness=row.pct_impervious
    )

    inp[INFILTRATION][row.SWMM_ID] = InfiltrationGreenAmpt(
        subcatchment=row.SWMM_ID,
        suction_head=row['Suction'], # (mm)
        hydraulic_conductivity=row['Ksat'], # (mm/hr)
        moisture_deficit_init=row['IMD']
    )

    # default runoff parameters
    inp[SUBAREAS][row.SWMM_ID] = SubArea(
        subcatchment=row.SWMM_ID,
        n_imperv=0.015,
        n_perv=0.15,
        storage_imperv=2.0, # (mm)
        storage_perv=5.0, # (mm)
        pct_zero=25.0
    )

In [ ]:
for i, row in nodes.iterrows():
    if row['node_type'] == 'outfall':
        inp[OUTFALLS][row.SWMM_ID] = Outfall(
            name=row.SWMM_ID,
            elevation=row['invert_elevation'],
            kind = 'FREE'
        )
    else:
        inp[JUNCTIONS][row.SWMM_ID] = (Junction(
            name=row.SWMM_ID,
            elevation=row['invert_elevation']
        ))

In [ ]:
for i, row in edges.iterrows():
    inp[CONDUITS][row.SWMM_ID] = Conduit(
        name=row.SWMM_ID,
        from_node=node_to_id[row.start_node],
        to_node=node_to_id[row.end_node],
        length=row.geometry.length,
        roughness=0.013, # Manning's n, mostly RCP AND PVC Pipes.
    )

    inp[XSECTIONS][row.SWMM_ID] = CrossSection(
        link=row.SWMM_ID,
        shape='CIRCULAR',
        height=row['diameter']
    )

In [ ]:
inp.to_file('watershed_model.inp')

In [ ]:
swmm5_run('watershed_model.inp')